# 1. Load Dataset

In [0]:
catalog = "workspace"
schema  = "bde"
volume  = "assignment2"

GREEN_DST = f"/Volumes/{catalog}/{schema}/{volume}/green"
YELLOW_DST = f"/Volumes/{catalog}/{schema}/{volume}/yellow"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, StringType
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [0]:
green_df  = spark.read.parquet(GREEN_DST)
display(green_df)

In [0]:
yellow_df = spark.read.parquet(YELLOW_DST)
display(yellow_df)

In [0]:
green_df.printSchema()

In [0]:
yellow_df.printSchema()

In [0]:
# Total rows count before cleaning
green_rows = green_df.count()
yellow_rows = yellow_df.count()
print(f"Green rows: {green_rows:,}")
print(f"Yellow rows: {yellow_rows:,}")

## 1.1 Explore & Clean Dataset

In [0]:
# 4.a Filter trips finishing before starting time
green_df = green_df.filter(F.col("lpep_dropoff_datetime") >= F.col("lpep_pickup_datetime"))
yellow_df = yellow_df.filter(F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))

In [0]:
# Display min-max pickup/dropoff time

from pyspark.sql.functions import min, max

green_df.select(
    min("lpep_pickup_datetime").alias("min_pickup"),
    max("lpep_dropoff_datetime").alias("max_dropoff")
).show()

yellow_df.select(
    min("tpep_pickup_datetime").alias("min_pickup"),
    max("tpep_dropoff_datetime").alias("max_dropoff")
).show()

In [0]:
from pyspark.sql.functions import col

# Define valid range
start_range = "2014-01-01 00:00:00"
end_range   = "2024-12-31 23:59:59"

green_invalid = green_df.filter(
    (col("lpep_pickup_datetime") < start_range) | (col("lpep_pickup_datetime") > end_range) |
    (col("lpep_dropoff_datetime") < start_range) | (col("lpep_dropoff_datetime") > end_range)
)
print(green_invalid.count())

yellow_invalid = yellow_df.filter(
    (col("tpep_pickup_datetime") < start_range) | (col("tpep_pickup_datetime") > end_range) |
    (col("tpep_dropoff_datetime") < start_range) | (col("tpep_dropoff_datetime") > end_range)
)
yellow_invalid.count()

In [0]:
# 4.b Remove invalid date range
green_df = green_df.filter(
    (col("lpep_pickup_datetime").between(start_range, end_range)) &
    (col("lpep_dropoff_datetime").between(start_range, end_range))
)

yellow_df = yellow_df.filter(
    (col("tpep_pickup_datetime").between(start_range, end_range)) &
    (col("tpep_dropoff_datetime").between(start_range, end_range))
)

In [0]:
# To avoid null values we fill all missing values with zeroes
# Fill null with 0
green_df = green_df.fillna(0)
yellow_df = yellow_df.fillna(0)

In [0]:
from pyspark.sql.functions import unix_timestamp

# Calculate speed based on trip distance and duration
green_df = green_df.withColumn(
    "trip_duration_hours",
    (unix_timestamp("lpep_dropoff_datetime") - unix_timestamp("lpep_pickup_datetime")) / 3600
).withColumn(
    "speed_mph",
    col("trip_distance") / col("trip_duration_hours")
)

yellow_df = yellow_df.withColumn(
    "trip_duration_hours",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600
).withColumn(
    "speed_mph",
    col("trip_distance") / col("trip_duration_hours")
)

In [0]:
# 4.c & 4.d Keep only trips with positive and realistic speed
green_df = green_df.filter(
    (col("speed_mph") >= 0) & (col("speed_mph") <= 200) & (col("trip_duration_hours") > 0)
)

yellow_df = yellow_df.filter(
    (col("speed_mph") >= 0) & (col("speed_mph") <= 200) & (col("trip_duration_hours") > 0)
)

In [0]:
# Check duration wise
green_df.select(
    min("trip_duration_hours").alias("min_duration"),
    max("trip_duration_hours").alias("max_duration")
).show()

yellow_df.select(
    min("trip_duration_hours").alias("min_duration"),
    max("trip_duration_hours").alias("max_duration")
).show()

In [0]:
# 4.e & f Remove trips with unrealistic duration and distance
green_df = green_df.filter(
    (col("trip_duration_hours") >= 0.0167) & (col("trip_duration_hours") <= 5.0) &  # 1 min to 5 hrs
    (col("trip_distance") >= 0.1) & (col("trip_distance") <= 100))    # 0.1 to 100 miles

yellow_df = yellow_df.filter(
    (col("trip_duration_hours") >= 0.0167) & (col("trip_duration_hours") <= 5.0) &  # 1 min to 5 hrs
    (col("trip_distance") >= 0.1) & (col("trip_distance") <= 100))    # 0.1 to 100 miles

In [0]:
# Sense check fare
green_df.select(
    min("fare_amount").alias("min_fare"),
    max("fare_amount").alias("max_fare")
).show()

yellow_df.select(
    min("fare_amount").alias("min_fare"),
    max("fare_amount").alias("max_fare")
).show()

In [0]:
# Sense check passenger count
green_df.select(
    min("passenger_count").alias("min_passenger_count"),
    max("passenger_count").alias("max_passenger_count")
).show()

yellow_df.select(
    min("passenger_count").alias("min_passenger_count"),
    max("passenger_count").alias("max_passenger_count")
).show()

In [0]:
# Base fare for every trip in NYC is $3
green_df = green_df.filter(
    (col("fare_amount") > 3) & (col("fare_amount") < 1000)
)
yellow_df = yellow_df.filter(
    (col("fare_amount") > 3) & (col("fare_amount") < 1000)
)

In [0]:
# Filter min passenger 1 and max 7
green_df = green_df.filter(
    (col("passenger_count") > 0) & (col("passenger_count") < 7)
)
yellow_df = yellow_df.filter(
    (col("passenger_count") > 0) & (col("passenger_count") < 7)
)

In [0]:
# Total rows count after cleaning
green_rows = green_df.count()
yellow_rows = yellow_df.count()
print(f"Green rows: {green_rows:,}")
print(f"Yellow rows: {yellow_rows:,}")

In [0]:
green_df.printSchema()

In [0]:
yellow_df.printSchema()

In [0]:
# Rename pickup/dropoff columns in yellow_df
yellow_df_renamed = yellow_df.withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
                             .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
green_df_renamed = green_df.withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
                           .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")

In [0]:
from pyspark.sql.functions import lit
# Align columns (add missing columns to each DataFrame)

# Columns to add to yellow_df to match green_df
yellow_df_aligned = yellow_df_renamed \
    .withColumn("trip_type", lit(None).cast("double")) \
    .withColumn("ehail_fee", lit(None).cast("double"))

# Columns to add to green_df to match yellow_df
green_df_aligned = green_df_renamed \
    .withColumn("airport_fee", lit(None).cast("double"))

In [0]:
columns_order = [
    "VendorID",
    "pickup_datetime",
    "dropoff_datetime",
    "store_and_fwd_flag",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "ehail_fee",
    "improvement_surcharge",
    "total_amount",
    "payment_type",
    "trip_type",
    "congestion_surcharge",
    "airport_fee",
    "trip_duration_hours",
    "speed_mph"
]

green_df_final = green_df_aligned.select(columns_order)
yellow_df_final = yellow_df_aligned.select(columns_order)

In [0]:
# Add indicator column
green_df_final = green_df_final.withColumn("taxi_type", lit("green"))
yellow_df_final = yellow_df_final.withColumn("taxi_type", lit("yellow"))

# Now union
combined_df = green_df_final.unionByName(yellow_df_final)

In [0]:
print("Green rows:", green_df_final.count())
print("Yellow rows:", yellow_df_final.count())
print("Combined rows:", combined_df.count())

In [0]:
taxi_zone_path = f"dbfs:/Volumes/workspace/bde/assignment2/taxi_zone_lookup.csv"

taxi_zones_df = spark.read.csv(taxi_zone_path, header=True, inferSchema=True)
taxi_zones_df.show(5)

In [0]:
# Pickup columns
pickup_zones = taxi_zones_df.select(
    col("LocationID").alias("PULocationID"),
    col("Zone").alias("pickup_zone"),
    col("Borough").alias("pickup_borough")
)

# Dropoff columns
dropoff_zones = taxi_zones_df.select(
    col("LocationID").alias("DOLocationID"),
    col("Zone").alias("dropoff_zone"),
    col("Borough").alias("dropoff_borough")
)

In [0]:
combined_df = combined_df.join(pickup_zones, on="PULocationID", how="left")
combined_df = combined_df.join(dropoff_zones, on="DOLocationID", how="left")

In [0]:
display(combined_df)

In [0]:
# Fill null with 0
combined_df = combined_df.fillna(0)

In [0]:
# More logic check
# Check extra column
combined_df.select(
    min("extra").alias("min_extra"),
    max("extra").alias("max_extra")
).show()

# Should be >0


In [0]:
from pyspark.sql.functions import avg, expr
# Check mta_tax column
combined_df.select(
    min("mta_tax").alias("min_mta_tax"),
    max("mta_tax").alias("max_mta_tax"),
    avg("mta_tax").alias("avg_mta_tax"),
    expr("percentile_approx(mta_tax, 0.5)").alias("median_mta_tax")
).show()


# mta_tax should be 0.50

In [0]:

# Check tip_amount column
combined_df.select(
    min("tip_amount").alias("min_tip_amount"),
    max("tip_amount").alias("max_tip_amount"),
    avg("tip_amount").alias("avg_tip_amount"),
    expr("percentile_approx(tip_amount, 0.5)").alias("median_tip_amount")
).show()

# Let's limit tip to <$50

In [0]:
# Count the number of unique payment types
combined_df.select("payment_type").distinct().show()

In [0]:
# Check congestion_surcharge
combined_df.select(
    min("congestion_surcharge").alias("min_congestion_surcharge"),
    max("congestion_surcharge").alias("max_congestion_surcharge")
).show()


In [0]:
# Check airport_fee
combined_df.select(
    min("airport_fee").alias("min_airport_fee"),
    max("airport_fee").alias("max_airport_fee")
).show()

# Check improvement_surcharge
combined_df.select(
    min("improvement_surcharge").alias("min_improvement_surcharge"),
    max("improvement_surcharge").alias("max_improvement_surcharge"),
    avg("improvement_surcharge").alias("avg_improvement_surcharge"),
    expr("percentile_approx(improvement_surcharge, 0.5)").alias("median_improvement_surcharge")
).show()

# improvement_surcharge should be $0.30

# Check ehail_fee
combined_df.select(
    min("ehail_fee").alias("min_ehail_fee"),
    max("ehail_fee").alias("max_ehail_fee")
).show()


In [0]:
# Define the condition
condition = (
    (col("tip_amount") >= 0) & (col("tip_amount") <= 50) &
    (col("extra") >= 0) &
    (col("mta_tax") >= 0) & (col("mta_tax") <= 0.5) &
    (col("improvement_surcharge") >= 0) & (col("improvement_surcharge") <= 1)
)

# Count rows that DON'T fit the condition
invalid_count = combined_df.filter(~condition).count()

print(f"Number of rows that will be filtered out: {invalid_count}")

In [0]:
# Apply data cleaning logic above
combined_df = combined_df.filter(condition)              

In [0]:
print("Combined rows:", combined_df.count())

In [0]:
# save as table
combined_df.write.format("delta").mode("overwrite").saveAsTable("cleaned_taxi_data")